# Section 1 — Data Quality Checks

Three factual questions built on `hotel_bookings.csv`.  
Every filter is tied to a specific footnote from the assessment brief.

---

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## Load Data

Two non-negotiable guards applied at load time:

- `keep_default_na=False` — preserves the literal string `'None'` in `customer_loyalty_tier`. Without this, pandas silently converts it to `NaN` and ~46% of tier data disappears **(Footnote 7)**.
- `review_rating` and `review_date` are nullable by design (empty when no review was left). They are converted separately using `errors='coerce'` so that empty strings do not corrupt the column dtype.
- Note: actual column name in this CSV is `checkin_date` (no underscore), not `check_in_date`.

In [ ]:
df = pd.read_csv(
    '../hotel_bookings (1).csv',
    keep_default_na=False,
    parse_dates=['customer_signup_date', 'booking_date', 'checkin_date', 'checkout_date']
)

# Nullable columns: empty strings -> NaN / NaT
df['review_rating'] = pd.to_numeric(df['review_rating'], errors='coerce')
df['review_date']   = pd.to_datetime(df['review_date'], errors='coerce')

print(f'Loaded: {len(df):,} rows x {df.shape[1]} columns')
df.dtypes

In [ ]:
# Quick sanity check: loyalty tier 'None' should be a string, not NaN
tier_counts = df['customer_loyalty_tier'].value_counts(dropna=False)
print('Loyalty tier distribution (None must appear as a string, not be missing):')
print(tier_counts)

---

## A1 — Invalid Stays Count

**Footnote 1:** Rows where `checkout_date <= checkin_date` are data errors.  
A valid hotel stay requires at minimum one night, meaning `checkout_date > checkin_date` must hold.

In [ ]:
invalid_stays = df[df['checkout_date'] <= df['checkin_date']]

print(f'A1 Answer — Invalid stays (checkout_date <= checkin_date): {len(invalid_stays):,}')
print()
invalid_stays[['booking_id', 'checkin_date', 'checkout_date', 'nights', 'booking_status']].head(10)

> **A1 Answer: 120 bookings** have `checkout_date <= checkin_date` and are treated as data errors.  
> These rows are excluded from any revenue, stay-duration, or occupancy calculation throughout the rest of the analysis.

---

## A2 — Review Rating: Range and Mean by Customer Segment

**Footnote 6:** `review_rating` uses **different scales per segment**:
- `Individual` and `Group` — scale **1 to 5**
- `Corporate` — scale **1 to 10**

Comparing means across segments without normalization will make Corporate appear artificially higher-rated.  
We show **raw stats** first (to expose the problem), then **normalized stats** (Corporate / 2 to bring everything onto a 1–5 scale).  

Only rows with an actual review are included — NaN `review_rating` rows are dropped.

In [ ]:
reviewed = df.dropna(subset=['review_rating'])

raw_stats = (
    reviewed
    .groupby('customer_segment')['review_rating']
    .agg(Min='min', Max='max', Mean='mean', Reviews='count')
    .round(2)
)

print('RAW stats — scale mismatch makes cross-segment mean comparison invalid:')
raw_stats

In [ ]:
# Normalize Corporate to 1-5 scale
reviewed_norm = reviewed.copy()
corp_mask = reviewed_norm['customer_segment'] == 'Corporate'
reviewed_norm.loc[corp_mask, 'review_rating'] = reviewed_norm.loc[corp_mask, 'review_rating'] / 2

norm_stats = (
    reviewed_norm
    .groupby('customer_segment')['review_rating']
    .agg(Min='min', Max='max', NormalizedMean='mean', Reviews='count')
    .round(2)
)

print('NORMALIZED stats — Corporate / 2, all segments now on 1-5 scale:')
norm_stats

**A2 Answer — Computed values:**

| Segment | Raw Min | Raw Max | Raw Mean | Normalized Mean (÷2 for Corporate) | Reviews |
|---|---|---|---|---|---|
| Corporate | 3.0 | 10.0 | 7.25 | 3.62 | 1,287 |
| Group | 1.0 | 5.0 | 3.77 | 3.77 | 590 |
| Individual | 1.0 | 5.0 | 3.77 | 3.77 | 3,269 |

> **1-line implication:** Corporate ratings are on a 1–10 scale while Individual and Group are on 1–5 — the raw Corporate mean (7.25) appears nearly double the others (3.77) purely due to the scale difference, not genuine satisfaction gap; after normalizing, all three segments are within 0.15 points of each other.

---

## A3 — Realized Revenue for Luxury Properties

Three footnotes must be applied **before** summing `total_amount`:

| Footnote | Issue | Action |
|---|---|---|
| **FN1** | Invalid stays (`checkout_date <= checkin_date`) | Exclude |
| **FN3** | Zero-room bookings (`num_rooms = 0`) | Exclude |
| **FN8** | Only `Completed` bookings count as realized revenue | Keep Completed only |

Two additional footnotes are data quality flags that do **not** change the revenue number but should be reported:
- **FN2** — bookings dated before customer signup date (temporal integrity violation)
- **FN5** — Cancelled bookings carrying a review rating (logically impossible)

After cleaning, filter `property_type == 'Luxury'` and sum `total_amount`.

In [ ]:
# FN2 and FN5 — flagged but not dropped for revenue
fn2_count = (df['booking_date'] < df['customer_signup_date']).sum()
fn5_count = ((df['booking_status'] == 'Cancelled') & df['review_rating'].notna()).sum()

print(f'FN2 — Bookings dated before customer signup  : {fn2_count:,} rows')
print(f'FN5 — Cancelled bookings with a review       : {fn5_count:,} rows')

In [ ]:
# Apply footnote filters in sequence, showing row count at each step
step1 = df[df['checkout_date'] > df['checkin_date']]       # FN1
step2 = step1[step1['num_rooms'] > 0]                       # FN3
step3 = step2[step2['booking_status'] == 'Completed']       # FN8
luxury = step3[step3['property_type'] == 'Luxury']          # target

realized_revenue = luxury['total_amount'].sum()

print(f'Starting rows (full dataset)              : {len(df):,}')
print(f'After FN1 (valid stays only)              : {len(step1):,}')
print(f'After FN3 (num_rooms > 0)                 : {len(step2):,}')
print(f'After FN8 (Completed only)                : {len(step3):,}')
print(f'After Luxury filter                       : {len(luxury):,}')
print()
print(f'Realized Revenue (Luxury) : Rs. {realized_revenue:,.2f}')

In [ ]:
# Show what the naive (no-filter) sum looks like vs. realized
naive             = df[df['property_type'] == 'Luxury']['total_amount'].sum()
overstatement     = naive - realized_revenue
overstatement_pct = (overstatement / naive) * 100

print(f'Naive sum (no footnote filters)  : Rs. {naive:,.2f}')
print(f'Realized revenue (with filters)  : Rs. {realized_revenue:,.2f}')
print(f'Overstatement                    : Rs. {overstatement:,.2f}  ({overstatement_pct:.1f}% of naive total)')

**A3 Answer — Computed values:**

| | Amount |
|---|---|
| Naive sum (no filters) | Rs. 1,17,43,148.47 |
| **Realized revenue (FN1 + FN3 + FN8 applied)** | **Rs. 90,69,405.293** |
| Overstatement without filters | Rs. 26,73,743.154 (22.8% of naive) |

> Realized revenue for `property_type = Luxury` is **Rs. 90,694,052.93**.  
> The naive sum overstates it by 22.8% — because Cancelled and No-Show bookings carry a `total_amount` that was never collected (FN8). This is the single biggest source of inflation.

---

## Section 1 — Final Answers Summary

| Question | Answer |
|---|---|
| **A1 — Invalid stays** | **120 bookings** with `checkout_date <= checkin_date` |
| **A2 — Rating implication** | Corporate raw mean is 7.25 (on 1–10) vs. 3.77 for others (on 1–5). After normalizing (Corporate ÷ 2), all segments fall between 3.62–3.77 — scale mismatch, not a real satisfaction difference |
| **A3 — Luxury realized revenue** | **Rs. 90,694,052.93** after applying FN1 + FN3 + FN8 (naive sum overstates by 22.8%) |

---

# Section 2 — Case Study: The Cancellation Crisis

**Briefing:** Platform cancel rate ~19.15%, industry benchmark 12%. Board target: reduce by 5 pp to ~14% within two quarters.

Four sub-questions: A1 landscape visualization, A2 rate vs. volume, A3 root-cause hypothesis testing, A4 targeted intervention.

---

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('../project/output', exist_ok=True)

MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

# FN1 + FN3 applied; FN8 NOT applied (need all statuses for cancel rate)
clean = df[
    (df['checkout_date'] > df['checkin_date']) &
    (df['num_rooms'] > 0)
].copy()

clean['checkin_month']  = clean['checkin_date'].dt.month
clean['lead_time_days'] = (clean['checkin_date'] - clean['booking_date']).dt.days
clean['is_cancelled']   = (clean['booking_status'] == 'Cancelled').astype(int)

bins   = [-1, 7, 30, 60, 90, 9999]
labels = ['0-7d', '8-30d', '31-60d', '61-90d', '90+d']
clean['lt_bucket'] = pd.cut(clean['lead_time_days'], bins=bins, labels=labels)

platform_rate   = clean['is_cancelled'].mean()
total_bookings  = len(clean)
total_cancelled = int(clean['is_cancelled'].sum())

print(f'Working dataset      : {total_bookings:,} rows')
print(f'Total cancelled      : {total_cancelled:,}')
print(f'Platform cancel rate : {platform_rate:.2%}')

---

## A1 — Cancellation Landscape

Two required visualizations:
1. **Heatmap** — `property_city` x calendar month, cell = cancel rate. Spots worst city-month instantly.
2. **Grouped bar** — lead-time bucket x booking channel, y = cancel rate. Tests whether short-notice bookings and specific channels independently drive cancellation.


In [ ]:
# A1 Viz 1: Cancellation Rate Heatmap (City x Month)
city_month = (
    clean.groupby(['property_city', 'checkin_month'])
    .agg(total=('booking_id', 'count'), cancelled=('is_cancelled', 'sum'))
    .assign(cancel_rate=lambda d: d['cancelled'] / d['total'])
    .reset_index()
)

heatmap_data = city_month.pivot(
    index='property_city', columns='checkin_month', values='cancel_rate'
)
heatmap_data.columns = [MONTH_NAMES[m - 1] for m in heatmap_data.columns]

fig, ax = plt.subplots(figsize=(15, 6))
sns.heatmap(
    heatmap_data, annot=True, fmt='.0%',
    cmap='YlOrRd', linewidths=0.5, linecolor='white', ax=ax
)
ax.set_title('A1 — Cancellation Rate by City x Month', fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Check-in Month', fontsize=11)
ax.set_ylabel('Property City', fontsize=11)
ax.tick_params(axis='both', rotation=0)
plt.tight_layout()
plt.savefig('../project/output/a1_cancellation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: project/output/a1_cancellation_heatmap.png')

In [ ]:
# A1 Viz 2: Cancel Rate by Lead-time Bucket x Booking Channel
lt_ch = (
    clean.groupby(['lt_bucket', 'booking_channel'], observed=True)
    .agg(total=('booking_id', 'count'), cancelled=('is_cancelled', 'sum'))
    .assign(cancel_rate=lambda d: d['cancelled'] / d['total'])
    .reset_index()
)

channels = sorted(clean['booking_channel'].unique())
x_pos    = range(len(labels))
width    = 0.2
colors   = ['#1976D2', '#E53935', '#43A047', '#8E24AA']

fig, ax = plt.subplots(figsize=(13, 5))
for i, ch in enumerate(channels):
    sub   = lt_ch[lt_ch['booking_channel'] == ch].set_index('lt_bucket')
    rates = [float(sub.loc[b, 'cancel_rate']) if b in sub.index else 0.0 for b in labels]
    ax.bar([xi + i * width for xi in x_pos], rates, width,
           label=ch, color=colors[i], alpha=0.88, edgecolor='white')

ax.set_xticks([xi + width * 1.5 for xi in x_pos])
ax.set_xticklabels(labels, fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_xlabel('Lead-time Bucket (days before check-in)', fontsize=11)
ax.set_ylabel('Cancellation Rate', fontsize=11)
ax.set_title('A1 — Cancel Rate by Lead-time x Booking Channel', fontsize=13, fontweight='bold', pad=12)
ax.legend(title='Channel', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../project/output/a1_leadtime_channel_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: project/output/a1_leadtime_channel_bar.png')

In [ ]:
# Which dimension drives most variance in cancel rate?
city_var  = city_month.groupby('property_city')['cancel_rate'].std().sort_values(ascending=False)
month_var = city_month.groupby('checkin_month')['cancel_rate'].std().sort_values(ascending=False)
lt_rates  = clean.groupby('lt_bucket', observed=True)['is_cancelled'].mean()

print('Std-dev of cancel rate BY CITY:')
for city, std in city_var.items():
    print(f'  {city:<18} {std:.4f}')

print(f'\nStd-dev of cancel rate BY MONTH (top 3):')
for m, std in list(month_var.items())[:3]:
    print(f'  {MONTH_NAMES[m-1]:<6} {std:.4f}')

print(f'\nCancel rate by lead-time bucket:')
for b, r in lt_rates.items():
    print(f'  {b}  {r:.1%}')

### A1 Answer

**Primary driver: City (specifically Goa)**
- Goa has the highest within-city cancel-rate std-dev across months: **0.1184** — more variance than any other city
- June has the highest cross-city std-dev: **0.1089** — more cross-city variance than any other month
- Lead-time adds a secondary signal: 31–60 day bucket has highest cancel rate (~23.8% vs 19.2% platform avg), but the spread is smaller than the city-month interaction

**Conclusion:** City × month interaction is the primary variance driver. Heatmap confirms Goa in Jun/Jul/Aug is the dominant hot zone. Lead-time is a secondary, platform-wide pattern.

In [ ]:
# A2 — Rate vs. Volume: two separate rankings
# Minimum 20 bookings per slice to avoid 100%-rate noise on tiny samples
significant = city_month[city_month['total'] >= 20].copy()
significant['month_name'] = significant['checkin_month'].apply(lambda m: MONTH_NAMES[m - 1])
significant['slice']      = significant['property_city'] + ' - ' + significant['month_name']

top_by_rate = significant.nlargest(3, 'cancel_rate')[['slice','cancel_rate','cancelled','total']].copy()
top_by_vol  = significant.nlargest(3, 'cancelled')[['slice','cancel_rate','cancelled','total']].copy()

for df_show, label in [(top_by_rate, 'TOP 3 BY CANCELLATION RATE (worst %)'),
                       (top_by_vol,  'TOP 3 BY ABSOLUTE CANCELLATION COUNT')]:
    tmp = df_show.copy()
    tmp['cancel_rate'] = tmp['cancel_rate'].map('{:.1%}'.format)
    tmp = tmp.rename(columns={'slice': 'City-Month', 'cancel_rate': 'Cancel Rate',
                               'cancelled': 'Cancellations', 'total': 'Total'})
    print(label)
    print(tmp.to_string(index=False))
    print()

### A2 Answer

**Top 3 by Cancellation Rate:**

| City-Month | Cancel Rate | Cancellations | Total Bookings |
|---|---|---|---|
| Goa - Jun | 50.9% | 28 | 55 |
| Goa - Aug | 44.7% | 38 | 85 |
| Goa - Jul | 36.9% | 24 | 65 |

**Top 3 by Absolute Cancellation Count:**

| City-Month | Cancel Rate | Cancellations | Total Bookings |
|---|---|---|---|
| Goa - Aug | 44.7% | 38 | 85 |
| Chennai - Aug | 22.2% | 37 | 167 |
| Pune - Feb | 24.0% | 37 | 154 |

**Why the distinction matters for the board's 5pp target:**

The high-rate and high-volume lists overlap but are NOT identical. Goa-Aug appears in both. But Chennai-Aug and Pune-Feb have much lower cancel rates (22–24%) yet produce the same absolute count (37 each) because their booking volumes are 2–3× larger.

The board's 5pp target is a platform-wide number. Targeting only Goa removes ~90 cancellations. Adding Chennai-Aug and Pune-Feb removes another ~50. Both tiers together are needed to close the gap.

---

## A3 — Root Cause: Hypothesis Testing

Worst-rate slice from A1/A2: **Goa in June (50.9% cancel rate, 28 of 55 bookings cancelled)**.

Three competing hypotheses tested with numbers:
- **H1** — Lead-time effect: is Goa-June booked unusually late (short-notice → higher cancel)?
- **H2** — Channel-mix effect: is OTA over-represented in Goa-June (OTA bookings cancel more)?
- **H3** — City-season effect: is this a genuine Goa × monsoon interaction, unique to this city-month pairing?

In [ ]:
# A3 — Root cause hypothesis testing on worst-rate slice
worst_row   = significant.loc[significant['cancel_rate'].idxmax()]
worst_city  = worst_row['property_city']
worst_month = int(worst_row['checkin_month'])
worst_rate  = worst_row['cancel_rate']

print(f'Worst slice  : {worst_city} in {MONTH_NAMES[worst_month - 1]}')
print(f'Cancel rate  : {worst_rate:.1%}  ({int(worst_row["cancelled"])} of {int(worst_row["total"])})')
print()

worst_mask = (clean['property_city'] == worst_city) & (clean['checkin_month'] == worst_month)
worst_df   = clean[worst_mask]
rest_df    = clean[~worst_mask]

# H1: Lead-time comparison
h1_worst = worst_df['lead_time_days'].median()
h1_rest  = rest_df['lead_time_days'].median()
print(f'H1 Median lead-time : {worst_city}-{MONTH_NAMES[worst_month-1]}={h1_worst:.0f}d  |  rest={h1_rest:.0f}d')

# H2: OTA share comparison
h2_worst_ota = (worst_df['booking_channel'] == 'OTA').mean()
h2_rest_ota  = (rest_df['booking_channel'] == 'OTA').mean()
print(f'H2 OTA share        : {worst_city}-{MONTH_NAMES[worst_month-1]}={h2_worst_ota:.1%}  |  rest={h2_rest_ota:.1%}')

# H3: City-season interaction (3-way comparison)
same_city_other = clean[(clean['property_city'] == worst_city) & (clean['checkin_month'] != worst_month)]
other_city_same = clean[(clean['property_city'] != worst_city) & (clean['checkin_month'] == worst_month)]

h3_same_city_other = same_city_other['is_cancelled'].mean()
h3_other_city_same = other_city_same['is_cancelled'].mean()

print(f'\nH3 City-season comparison:')
print(f'  {worst_city} in {MONTH_NAMES[worst_month-1]} (worst)        : {worst_rate:.1%}')
print(f'  {worst_city} in other months             : {h3_same_city_other:.1%}')
print(f'  Other cities in {MONTH_NAMES[worst_month-1]}               : {h3_other_city_same:.1%}')
print(f'  Platform average                         : {platform_rate:.1%}')

In [ ]:
# A3 — Hypothesis Comparison Chart (3 panels)
RED, BLUE, AMBER = '#E53935', '#1976D2', '#FB8C00'

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle(
    f'A3 Root Cause — {worst_city} in {MONTH_NAMES[worst_month-1]} ({worst_rate:.1%} cancel rate)',
    fontsize=13, fontweight='bold'
)

# Panel 1: H1 Lead-time
axes[0].bar(['Worst Slice', 'Rest'], [h1_worst, h1_rest],
            color=[RED, BLUE], alpha=0.85, width=0.5)
axes[0].set_title('H1: Median Lead-time', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Days')
axes[0].set_ylim(0, max(h1_worst, h1_rest) * 1.35)
for bar, val in zip(axes[0].patches, [h1_worst, h1_rest]):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.3, f'{val:.0f}d', ha='center', fontsize=12, fontweight='bold')
axes[0].text(0.5, 0.93, 'RULED OUT', transform=axes[0].transAxes,
             ha='center', color='green', fontsize=9, fontweight='bold')

# Panel 2: H2 OTA share
axes[1].bar(['Worst Slice', 'Rest'], [h2_worst_ota, h2_rest_ota],
            color=[RED, BLUE], alpha=0.85, width=0.5)
axes[1].set_title('H2: OTA Channel Share', fontsize=11, fontweight='bold')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].set_ylim(0, max(h2_worst_ota, h2_rest_ota) * 1.35)
for bar, val in zip(axes[1].patches, [h2_worst_ota, h2_rest_ota]):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.004, f'{val:.1%}', ha='center', fontsize=12, fontweight='bold')
axes[1].text(0.5, 0.93, 'RULED OUT', transform=axes[1].transAxes,
             ha='center', color='green', fontsize=9, fontweight='bold')

# Panel 3: H3 City-season
NL = chr(10)
h3_vals   = [worst_rate, h3_same_city_other, h3_other_city_same, platform_rate]
h3_labels = [
    f'{worst_city}{NL}{MONTH_NAMES[worst_month-1]}{NL}(Worst)',
    f'{worst_city}{NL}Other Months',
    f'Other Cities{NL}{MONTH_NAMES[worst_month-1]}',
    f'Platform{NL}Avg'
]
h3_colors = [RED, AMBER, AMBER, BLUE]
axes[2].bar(h3_labels, h3_vals, color=h3_colors, alpha=0.85, width=0.5)
axes[2].set_title('H3: City-Season Interaction', fontsize=11, fontweight='bold')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[2].set_ylim(0, max(h3_vals) * 1.35)
for bar, val in zip(axes[2].patches, h3_vals):
    axes[2].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.004, f'{val:.1%}', ha='center', fontsize=10, fontweight='bold')
axes[2].text(0.5, 0.93, 'SUPPORTED', transform=axes[2].transAxes,
             ha='center', color='red', fontsize=9, fontweight='bold')

for ax in axes:
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../project/output/a3_hypothesis_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: project/output/a3_hypothesis_comparison.png')

### A3 Answer

| Hypothesis | Metric | Goa-Jun (Worst) | Rest of Platform | Verdict |
|---|---|---|---|---|
| H1 — Lead-time | Median lead-time | 15 days | 15 days | **Ruled out** — identical |
| H2 — Channel mix | OTA share | 36.4% | 35.4% | **Ruled out** — negligible gap |
| H3 — City-season | Cancel rate vs. comparators | 50.9% | Goa other months: 24.4% / Other cities June: 19.4% | **Supported** |

**Conclusion — H3 is the only supported hypothesis.**

Goa in June cancels at 50.9% — **2× its own annual average** (24.4%) and **2.6× other cities in the same month** (19.4%). Lead-time is identical (15d vs 15d) and OTA share is essentially equal. This is a genuine **city-season interaction driven by monsoon uncertainty** — leisure travelers book Goa ahead of the season, then cancel when monsoon forecasts deteriorate. Channel and advance notice do not explain it; the destination-season pairing does.

In [ ]:
# A4 — Quantified Impact Math
# Target: Goa Jun-Aug (full monsoon window, not just June)
monsoon_months = [6, 7, 8]
target      = clean[(clean['property_city'] == worst_city) &
                     (clean['checkin_month'].isin(monsoon_months))]
t_bookings  = len(target)
t_cancelled = int(target['is_cancelled'].sum())
t_rate      = t_cancelled / t_bookings

print(f'Target: {worst_city} | Jun-Aug (monsoon season)')
print(f'  Bookings    : {t_bookings}  ({t_bookings/total_bookings:.1%} of platform)')
print(f'  Cancelled   : {t_cancelled}  ({t_cancelled/total_cancelled:.1%} of platform cancellations)')
print(f'  Cancel rate : {t_rate:.1%}')
print()

for pct, label in [(0.30, 'Conservative (30% cancel reduction)'),
                   (0.50, 'Optimistic  (50% cancel reduction)')]:
    saved         = t_cancelled * pct
    new_cancelled = total_cancelled - saved
    new_rate      = new_cancelled / total_bookings
    pp_impact     = (platform_rate - new_rate) * 100
    print(f'[{label}]')
    print(f'  Cancellations prevented  : {saved:.0f}')
    print(f'  New platform cancel rate : {new_rate:.2%}')
    print(f'  Platform-wide reduction  : {pp_impact:.2f} pp')
    print()

### A4 Answer — Board-Level Recommendation

---

**WHO:** All bookings for Goa properties with check-in in **June, July, or August** (monsoon season — 205 bookings, 90 cancellations, 43.9% avg cancel rate).

**WHAT:** Require a non-refundable **20% deposit** at booking confirmation for this segment. Offer an optional **Monsoon Flex add-on at Rs. 499** that restores full refundability for cancellations ≥ 7 days before check-in. Converts uncertain bookings into committed revenue while giving price-sensitive customers an opt-out.

**QUANTIFIED IMPACT:**

| Scenario | Reduction in Goa monsoon cancellations | Prevented | New platform rate | Platform-wide reduction |
|---|---|---|---|---|
| Conservative | 30% of 90 | ~27 | ~18.92% | ~0.23 pp |
| Optimistic | 50% of 90 | ~45 | ~18.77% | ~0.38 pp |

> This single intervention closes **0.23–0.38 pp** of the 5 pp gap. Reaching 14% requires complementary policies for Chennai-Aug (37 cancellations, 22.2%) and Pune-Feb (37 cancellations, 24.0%). Combined three-slice impact: ~100 cancellations prevented, ~**0.85 pp** reduction.

**RISK:** Deposit may suppress new Goa monsoon bookings from price-sensitive leisure travelers. If OTA conversion in Goa Jun-Aug drops >15%, lost booking value outweighs cancellation savings. **Mitigation:** A/B test on 50% of Goa monsoon traffic for 4 weeks; monitor OTA conversion before full rollout.

---

## Section 2 — Final Answers Summary

| Question | Answer |
|---|---|
| **A1 — Primary dimension** | City (Goa, std 0.1184); Goa Jun/Jul/Aug is the dominant cancellation hot zone |
| **A2 — Rate vs. Volume** | High-rate = Goa Jun/Jul/Aug; High-volume = Goa-Aug + Chennai-Aug + Pune-Feb. Volume matters more for the board's 5pp platform-wide target |
| **A3 — Root cause** | H3 city-season confirmed. H1 (lead-time identical: 15d=15d) and H2 (OTA share: 36.4% vs 35.4%) ruled out. Monsoon uncertainty drives Goa cancellations |
| **A4 — Recommendation** | 20% non-refundable deposit for Goa Jun-Aug. Expected impact: **0.23–0.38 pp** platform-wide. Risk: leisure conversion suppression on OTA |